# **Task 3: AI Resume Screening System with Tracing**

An AI-powered Resume Screening System is a GenAI-based application that automates the process of evaluating candidate resumes against a job description. It extracts relevant skills, compares them with required job skills, assigns a suitability score and provides an explanation for the evaluation. This helps recruiters quickly identify the best-fit candidates in a structured and consistent way using a machine learning pipeline built with LangChain.

##Objective

The main objectives of this project are:

* To build an automated resume screening pipeline using Generative AI techniques
* To extract key skills, experience, and tools from resumes
* To compare extracted data with job requirements for matching
* To generate a quantitative fit score (0–100) for each candidate
* To provide clear and explainable reasoning for the assigned score
* To implement a modular LLM-based workflow using prompt engineering and chaining
* To understand real-world application of AI in recruitment systems


**Install libraries**

In [ ]:
!pip install -q langchain langchain-core langchain-community langchain-huggingface huggingface_hub langsmith transformers

In [ ]:
!pip install -q transformers langchain langchain-core

In [ ]:
!pip install requests==2.32.4

**Set Environment Variables**

In [ ]:
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "your hf key here"

os.environ["LANGCHAIN_API_KEY"] = "your lc key here"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening"

**Import Libraries**

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_huggingface import HuggingFaceEndpoint

**Create LLM**

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

hf_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

**Input Data**

In [ ]:
job_description = """
Python, Machine Learning, SQL, Pandas, Scikit-learn, Data Visualization
"""

strong_resume = "Python, Machine Learning, SQL, Pandas, Scikit-learn, AWS"
average_resume = "Python, SQL, Excel, Data Visualization"
weak_resume = "MS Word, PowerPoint, Excel"

**Step 1: Extraction**

In [ ]:
extract_prompt = PromptTemplate.from_template("""
Extract skills from resume.

Resume:
{resume}
""")

extract_chain = extract_prompt | llm

**Step 2: Matching Logic**

In [ ]:
job_skills = job_description.lower().split(",")

def match_logic(data):
    resume = data["skills"].lower()

    matched = [s for s in job_skills if s.strip() in resume]
    missing = [s for s in job_skills if s.strip() not in resume]

    return {
        "matched": matched,
        "missing": missing
    }

match_chain = RunnableLambda(match_logic)

**Step 3: Scoring**

In [ ]:
def score_logic(data):
    total = len(job_skills)
    score = int((len(data["matched"]) / total) * 100)

    return {
        **data,
        "score": score
    }

score_chain = RunnableLambda(score_logic)

**Step 4: Explanation**

In [ ]:
def explain_logic(data):
    return f"""
Score: {data['score']}

Matched: {data['matched']}
Missing: {data['missing']}

Candidate suitability based on skill overlap.
"""

explain_chain = RunnableLambda(explain_logic)

**FULL PIPELINE**

In [ ]:
def pipeline(resume):

    step1 = extract_chain.invoke({"resume": resume})

    step2 = match_chain.invoke({"skills": step1})

    step3 = score_chain.invoke(step2)

    final = explain_chain.invoke(step3)

    print(final)

In [ ]:
print("===== STRONG =====")
pipeline(strong_resume)

print("\n===== AVERAGE =====")
pipeline(average_resume)

print("\n===== WEAK =====")
pipeline(weak_resume)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== STRONG =====


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score: 83

Matched: ['\npython', ' machine learning', ' sql', ' pandas', ' scikit-learn']
Missing: [' data visualization\n']

Candidate suitability based on skill overlap.


===== AVERAGE =====


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score: 50

Matched: ['\npython', ' sql', ' data visualization\n']
Missing: [' machine learning', ' pandas', ' scikit-learn']

Candidate suitability based on skill overlap.


===== WEAK =====

Score: 0

Matched: []
Missing: ['\npython', ' machine learning', ' sql', ' pandas', ' scikit-learn', ' data visualization\n']

Candidate suitability based on skill overlap.

